# Cross-Architectural Cartoonization Robustness & Perceptual Benchmark
### Contributor: Saiswaroop
This notebook contains the complete pipeline for evaluating cartoonization models across Vision Transformers (ViT-B/16), progressive JPEG compression (Uncompressed vs Q50), and LPIPS perceptual feature drift.

In [ ]:
# 1. Install modern diffusion, vision transformers, and LPIPS
!pip install -q diffusers transformers accelerate timm lpips

import torch
import os

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"--> Hardware Active: {device}")
if torch.cuda.is_available():
    print(f"--> GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"--> Device Count: {torch.cuda.device_count()}")

In [ ]:
# 2. High-speed dataset download
!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
!tar -xzf imagenette2-320.tgz
print("Imagenette-320 downloaded and extracted successfully!")

In [ ]:
# 3. Load AnimeGANv2 and define clean OpenCV pipeline
import cv2
import numpy as np
from PIL import Image
import torchvision.transforms as transforms

# Clean OpenCV cartoonization (C=7, blockSize=15 to eliminate noise)
def cartoonize_opencv_clean(pil_img):
    img_np = np.array(pil_img.convert('RGB'))
    color = cv2.bilateralFilter(img_np, d=9, sigmaColor=75, sigmaSpace=75)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    gray_blur = cv2.medianBlur(gray, 7)
    edges = cv2.adaptiveThreshold(gray_blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, 7)
    edges_color = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(cv2.bitwise_and(color, edges_color))

# Load AnimeGANv2 generator on GPU
animegan = torch.hub.load("bryandlee/animegan2-pytorch:main", "generator", pretrained="face_paint_512_v2", device=device).eval()

def stylize_animegan(pil_img):
    img = pil_img.convert("RGB").resize((256, 256))
    img_t = transforms.ToTensor()(img).unsqueeze(0).to(device) * 2 - 1
    with torch.no_grad():
        out = animegan(img_t)
        out = (out.squeeze().cpu().clamp(-1, 1) + 1) / 2
    return transforms.ToPILImage()(out)

In [ ]:
# 4. Run ViT-B/16 with 10-Class Logit Subsetting
import io
import glob
from tqdm import tqdm
from torchvision import models

IMAGENETTE_CLASSES = {
    "n01440764": {"idx": 0,   "name": "Tench (fish)"},
    "n02102040": {"idx": 217, "name": "English springer"},
    "n02979186": {"idx": 482, "name": "Cassette player"},
    "n03000684": {"idx": 491, "name": "Chain saw"},
    "n03028079": {"idx": 497, "name": "Church"},
    "n03394916": {"idx": 566, "name": "French horn"},
    "n03425413": {"idx": 569, "name": "Garbage truck"},
    "n03444034": {"idx": 571, "name": "Gas pump"},
    "n03445777": {"idx": 574, "name": "Golf ball"},
    "n03888257": {"idx": 701, "name": "Parachute"}
}

target_indices = [info["idx"] for info in IMAGENETTE_CLASSES.values()]
target_tensor = torch.tensor(target_indices, device=device)

weights = models.ViT_B_16_Weights.DEFAULT
vit_model = models.vit_b_16(weights=weights).to(device).eval()
val_transforms = weights.transforms()

def compress_jpeg(pil_img, quality):
    if quality == "uncompressed":
        return pil_img
    buf = io.BytesIO()
    pil_img.save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return Image.open(buf)

In [ ]:
# 5. Measure LPIPS Perceptual Feature Drift
import lpips
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device).eval()

t_lpips = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def compute_lpips(img1_pil, img2_pil):
    t1 = t_lpips(img1_pil.convert('RGB')).unsqueeze(0).to(device)
    t2 = t_lpips(img2_pil.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        dist = loss_fn_vgg(t1, t2).item()
    return dist

sample_images = glob.glob("imagenette2-320/val/*/*.JPEG")[:50]
opencv_lpips, animegan_lpips = [], []

for path in sample_images:
    orig = Image.open(path).convert('RGB')
    cv_img = cartoonize_opencv_clean(orig)
    opencv_lpips.append(compute_lpips(orig, cv_img))
    an_img = stylize_animegan(orig)
    animegan_lpips.append(compute_lpips(orig, an_img))

print(f"--> OpenCV Mean LPIPS   : {sum(opencv_lpips)/len(opencv_lpips):.4f}")
print(f"--> AnimeGAN Mean LPIPS : {sum(animegan_lpips)/len(animegan_lpips):.4f}")